# Webinar 2: Data Preprocessing — Track 2: Text Pipeline
### Dataset: 20 Newsgroups (Real Internet Forum Discussions — sci.med vs alt.atheism)
### Algorithms: LinearSVC (SVM) vs Baseline SGD Classifier

This notebook covers the complete NLP text preprocessing workflow on real discussion posts:
1. **Text Cleaning**: Removing email artifacts, quotation headers, special characters, and stopwords.
2. **TF-IDF Vectorization**: Extracting unigram + bigram representations with sublinear TF scaling.
3. **Numerical Feature Engineering**: Extracting text length, uppercase shouting ratio, punctuation density.
4. **Imbalance Handling**: Addressing class imbalance using SMOTE on text embeddings.
5. **Model Evaluation**: Comparing raw Bag-of-Words SGD Classifier vs Cleaned TF-IDF **LinearSVC**.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.text import (
    clean_text,
    batch_clean_texts,
    TFIDFProcessor,
    extract_numerical_text_features,
    run_text_pipeline
)

print('Text NLP modules imported successfully!')

## Step 1: Inspect Real Internet Forum Posts
Observing real-world noise: quotation tags, email fragments, contractions, and typos.

In [ ]:
df_text = pd.read_csv('../data/text/newsgroups_raw.csv')
print(f'Total documents: {len(df_text)}')
df_text.head()

## Step 2: Cleaning & Text Normalization

In [ ]:
sample_raw = df_text['raw_text'].iloc[0]
sample_cleaned = clean_text(sample_raw)
print(f'BEFORE CLEANING (first 250 chars):\n{sample_raw[:250]}...\n')
print(f'AFTER CLEANING (first 250 chars):\n{sample_cleaned[:250]}...')

## Step 3: TF-IDF Vectorization with N-Grams

In [ ]:
cleaned_texts = batch_clean_texts(df_text['raw_text'])
tfidf = TFIDFProcessor(max_features=500, ngram_range=(1, 2), sublinear_tf=True)
tfidf_matrix = tfidf.fit_transform(cleaned_texts)
print(f'TF-IDF Matrix Shape: {tfidf_matrix.shape}')
top_kw = tfidf.get_top_keywords(cleaned_texts, top_n=10)
top_kw

## Step 4: Linguistic Numerical Feature Engineering

In [ ]:
num_feats = extract_numerical_text_features(df_text['raw_text'])
num_feats.head()

## Step 5: Full NLP Pipeline Execution (LinearSVC vs SGD)

In [ ]:
text_results = run_text_pipeline('../data/text/newsgroups_raw.csv')

from src.evaluation.comparison import generate_modality_comparison
comp_df = generate_modality_comparison(text_results['baseline_metrics'], text_results['preprocessed_metrics'], 'Text (20 Newsgroups)')
comp_df